# PRT565 Assessment 3 — Heart Disease Classification

**Project:** Comparing Decision Tree, Random Forest, Naïve Bayes and Multilayer ANN for Heart Disease Prediction

This notebook uses the **UCI Heart Disease (Cleveland) dataset**, converts the original `num` target to binary classification (`0 = no disease`, `1 = disease present`), preprocesses the data, trains the four models required by the assessment, and compares Accuracy, Precision, Recall and F1-score.

**Reproducibility:** fixed `random_state=42`; 80/20 stratified train/test split.

## 1. Install/import libraries
Google Colab already contains most packages. This notebook uses pandas, NumPy, matplotlib, seaborn and scikit-learn.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)
print("Libraries loaded successfully.")

## 2. Load the UCI Cleveland Heart Disease dataset
UCI reports 303 instances and 13 predictive features. The raw target `num` ranges from 0 to 4; UCI notes that experiments commonly distinguish absence (`0`) from presence (`1–4`) of heart disease.

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
columns = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"
]

df = pd.read_csv(url, header=None, names=columns, na_values="?")
print("Original shape:", df.shape)
display(df.head())

## 3. Data understanding and cleaning

In [ ]:
print(df.info())
print("\nMissing values by column:")
display(df.isna().sum().to_frame("missing"))
print("\nRaw target distribution:")
print(df["num"].value_counts().sort_index())

In [ ]:
# Convert target to binary: 0 = no disease, 1 = disease present
df["target"] = (df["num"] > 0).astype(int)
df = df.drop(columns="num")

# We keep all 13 predictor variables and handle missing predictor values
# inside each model pipeline using median imputation.
print("Working shape:", df.shape)
print("Binary target distribution:")
print(df["target"].value_counts())
display(df.describe().T)

## 4. Exploratory data analysis

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="target")
plt.xticks([0,1], ["No disease", "Disease present"])
plt.title("Heart Disease Class Distribution")
plt.xlabel("Target")
plt.ylabel("Patients")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(11,8))
sns.heatmap(df.corr(numeric_only=True), cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))
sns.boxplot(data=df, x="target", y="age", ax=axes[0])
axes[0].set_title("Age by Target")
sns.boxplot(data=df, x="target", y="thalach", ax=axes[1])
axes[1].set_title("Max Heart Rate by Target")
sns.boxplot(data=df, x="target", y="oldpeak", ax=axes[2])
axes[2].set_title("ST Depression by Target")
plt.tight_layout()
plt.show()

## 5. Train/test split and preprocessing

In [ ]:
X = df.drop(columns="target")
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

### Preprocessing design
- Missing predictor values: median imputation fitted on training data only.
- Decision Tree and Random Forest: imputation only.
- Gaussian Naïve Bayes and Multilayer ANN: imputation + standardisation.
- This prevents test-set information leaking into preprocessing.

## 6. Define and train the four required models

In [ ]:
models = {
    "Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE))
    ]),
    "Naive Bayes": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", GaussianNB())
    ]),
    "Multilayer ANN": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", MLPClassifier(hidden_layer_sizes=(32,16), activation="relu",
                                solver="adam", alpha=0.001, max_iter=3000,
                                early_stopping=True, random_state=RANDOM_STATE))
    ])
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(name, "trained")

## 7. Evaluate Accuracy, Precision, Recall and F1-score

In [ ]:
results = []
predictions = {}
for name, model in models.items():
    pred = model.predict(X_test)
    predictions[name] = pred
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-Score": f1_score(y_test, pred, zero_division=0)
    })

results_df = pd.DataFrame(results).sort_values("F1-Score", ascending=False).reset_index(drop=True)
display(results_df.style.format({c: "{:.3f}" for c in ["Accuracy","Precision","Recall","F1-Score"]}))
results_df.to_csv("model_results.csv", index=False)

In [ ]:
plot_df = results_df.melt(id_vars="Model", var_name="Metric", value_name="Score")
plt.figure(figsize=(10,5))
sns.barplot(data=plot_df, x="Model", y="Score", hue="Metric")
plt.ylim(0,1)
plt.title("Model Performance Comparison")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("model_performance.png", dpi=200, bbox_inches="tight")
plt.show()

## 8. Confusion matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10,8))
for ax, (name, pred) in zip(axes.ravel(), predictions.items()):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=200, bbox_inches="tight")
plt.show()

## 9. Detailed classification reports

In [ ]:
for name, pred in predictions.items():
    print("\n" + "="*65)
    print(name)
    print("="*65)
    print(classification_report(y_test, pred, target_names=["No disease", "Disease"], zero_division=0))

## 10. Random Forest feature importance

In [ ]:
rf = models["Random Forest"].named_steps["model"]
importance = pd.DataFrame({"Feature": X.columns, "Importance": rf.feature_importances_})
importance = importance.sort_values("Importance", ascending=False)
display(importance)

plt.figure(figsize=(8,5))
sns.barplot(data=importance.head(10), x="Importance", y="Feature")
plt.title("Top 10 Random Forest Feature Importances")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=200, bbox_inches="tight")
plt.show()

## 11. Automatic conclusion based on the notebook results

In [ ]:
best = results_df.iloc[0]
print(f"Highest F1-score in this fixed test split: {best['Model']} ({best['F1-Score']:.3f}).")
print(f"Its accuracy was {best['Accuracy']:.3f}, precision {best['Precision']:.3f}, and recall {best['Recall']:.3f}.")
print("Because the dataset is small, these test-set results should be interpreted as a comparison for this experiment rather than a clinical performance guarantee.")

## 12. References (APA 7)

Janosi, A., Steinbrunn, W., Pfisterer, M., & Detrano, R. (1989). *Heart Disease* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C52P4X

Detrano, R., Janosi, A., Steinbrunn, W., Pfisterer, M., Schmid, J.-J., Sandhu, S., Guppy, K. H., Lee, S., & Froelicher, V. (1989). International application of a new probability algorithm for the diagnosis of coronary artery disease. *The American Journal of Cardiology, 64*(5), 304–310.

Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., Blondel, M., Prettenhofer, P., Weiss, R., Dubourg, V., Vanderplas, J., Passos, A., Cournapeau, D., Brucher, M., Perrot, M., & Duchesnay, É. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830.